# Create target list for the BGS Extension spare fiber program

# Setup:
Ensure that this notebook is running on the kernel with the legacy survey cutout service + dask docker image.

To set up the Dask cluster, run following cell in Jupyter terminal

`salloc -N 4 -n 512 -t 240 -C cpu -q interactive --image=biprateep/desi-dask:latest --account=desi`

`salloc -N 4 -n 4 -c 256 -t 240 -C cpu -q interactive --image=biprateep/desi-dask:latest --account=desi`
and then 

`./launch_dask.sh` 

the `-n` argument controls the number of workers to be launched.

In [1]:
import os
import shutil
from pathlib import Path
from importlib import reload


import numpy as np
import pandas as pd
from astropy.table import Table, vstack, hstack
import fitsio
import dask
from dask.distributed import Client
import dask.dataframe

from desitarget.targets import encode_targetid
from desimodel.footprint import radec2pix
import matplotlib.pyplot as plt

Matplotlib created a temporary cache directory at /tmp/matplotlib-4exmg11b because the default path (/homedir/.config/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
# figure defaults for AASTEX AJ

COLUMN_WIDTH = 242.26653/72.27 #in inches
TEXT_WIDTH = 513.11743/72.27
SMALL_SIZE = 9 # in pts
NORMAL_SIZE = 10 
BIG_SIZE = 12
FONT_FAMILY = 'Nimbus Roman No9 L'



params = {
    "font.family": FONT_FAMILY,
    "font.size": NORMAL_SIZE,
    "axes.titlesize": NORMAL_SIZE,
    "axes.labelsize": NORMAL_SIZE,
    
    "xtick.labelsize": SMALL_SIZE,
    "ytick.labelsize": SMALL_SIZE,
    "xtick.top": True,
    "ytick.right": True,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "legend.fontsize": NORMAL_SIZE,
    
    "figure.facecolor": "w",
    "figure.dpi": 300,
    'mathtext.fontset' : "cm"
    
}

plt.rcParams.update(params)

In [3]:
# Read the scheduler file generated by the script above and connect your notebook to the client
scheduler_file = os.path.join(os.environ["SCRATCH"], "scheduler.json")
# scheduler_file = os.path.join(os.environ["CFS"],'desi','users','bid13', "scheduler.json")
dask.config.config["distributed"]["dashboard"]["link"] = "{JUPYTERHUB_SERVICE_PREFIX}proxy/{host}:{port}/status"
client = Client(scheduler_file=scheduler_file)
client

<Client: 'tcp://10.249.1.65:8786' processes=64 threads=256, memory=1.63 TiB>

In [4]:
cat_path = Path("/global/cfs/cdirs/cosmo/data/legacysurvey/dr11/")
dest_path = Path(os.environ["SCRATCH"]) / "photo_z_spare_fiber"


### Dask delayed function to read and select required galaxies

In [5]:
# i-band columns are present only in the DR11 south (DECam) sweeps, not in the
# north (BASS/MzLS) ones. Drop them so that every partition handed to
# dask.dataframe.from_delayed has the same schema.
I_BAND_COLS = ['FLUX_I', 'FLUX_IVAR_I', 'MW_TRANSMISSION_I', 'NOBS_I', 'RCHISQ_I',
               'FRACFLUX_I', 'FRACMASKED_I', 'FRACIN_I', 'ANYMASK_I', 'ALLMASK_I',
               'PSFSIZE_I', 'PSFDEPTH_I', 'GALDEPTH_I', 'FIBERFLUX_I', 'FIBERTOTFLUX_I']

# Sweeps are read in row blocks rather than whole: DR11 south files run to 7 GB,
# and reading one entire file into an astropy Table costs ~2.2x the file size.
# Row blocks keep the per-task footprint flat (~0.5 GB) whatever the file size.
ROW_CHUNK = 250_000


def select_targets(sweep, survey):
    """Apply the BGS extension cuts to one block of rows of a sweep file."""
    sweep.remove_column('DCHISQ')
    sweep.remove_columns([c for c in I_BAND_COLS if c in sweep.colnames])
    sweep["PHOTSYS"] = survey[0].upper()
    sweep["MYID"] = encode_targetid(objid = sweep["OBJID"],
                                    brickid = sweep["BRICKID"],
                                    release = sweep["RELEASE"])
    sweep["HEALPIX"] = radec2pix(nside = 64, ra = sweep["RA"], dec = sweep["DEC"])


    GFLUX = sweep["FLUX_G"] / sweep["MW_TRANSMISSION_G"]
    RFLUX = sweep["FLUX_R"] / sweep["MW_TRANSMISSION_R"]
    ZFLUX = sweep["FLUX_Z"] / sweep["MW_TRANSMISSION_Z"]
    W1FLUX = sweep["FLUX_W1"] / sweep["MW_TRANSMISSION_W1"]
    W1SNR = sweep["FLUX_W1"] * np.sqrt(sweep["FLUX_IVAR_W1"])
    G = sweep["GAIA_PHOT_G_MEAN_MAG"]
    Grr = sweep["GAIA_PHOT_G_MEAN_MAG"] - 22.5 + 2.5*np.log10(sweep["FLUX_R"])
    RFIBREFLUX = sweep["FIBERFLUX_R"]/ sweep["MW_TRANSMISSION_R"]
    rmag = 22.5 - 2.5*np.log10(np.clip(RFLUX, 1e-16, None))
    zmag = 22.5 - 2.5*np.log10(np.clip(ZFLUX, 1e-16, None))
    gmag = 22.5 - 2.5*np.log10(np.clip(GFLUX, 1e-16, None))
    w1mag = 22.5 - 2.5*np.log10(np.clip(W1FLUX, 1e-16, None))
    rfibmag = 22.5 - 2.5*np.log10(np.clip(RFIBREFLUX, 1e-16, None))
    rfibtotmag = 22.5 - 2.5*np.log10(np.clip(sweep["FIBERTOTFLUX_R"], 1e-16, None))



    #Valid photometry
    QC =  (sweep["FLUX_IVAR_G"] > 0) & (sweep["FLUX_IVAR_R"] > 0) & (sweep["FLUX_IVAR_Z"] > 0)



    #color range sanity
    QC &= ((gmag-rmag) > -1) & ((gmag-rmag) < 4)
    QC &= ((rmag-zmag) > -1) & ((rmag-zmag) < 4)

    # Atleast one observation
    QC &= (sweep["NOBS_G"] > 0) & (sweep["NOBS_R"] > 0) & (sweep["NOBS_Z"] > 0) 

    #Star galaxy separation
    #Will apply the TYPE!= PSF cut later on for easier reconstruction of parent sample
    # GALAXY = ((zmag - w1mag) > 0.8 * (rmag - zmag) - 1) | ((rmag - zmag) < 0.6) # Discarding in favor of PSF, W1 detection may throw out star forming galaxies
    # GALAXY = sweep["TYPE"] != "PSF"




    #Bright star and globular cluster masks        
    MASKBITS = ((sweep["MASKBITS"] & 2**1) == 0) #touches bright star
    MASKBITS &= ((sweep["MASKBITS"] & 2**13) == 0) #touches globular cluster
    #(should we mask SGA?)
    MASKBITS &= ((sweep["MASKBITS"] & 2**12) == 0) #touches SGA




    MAGLIM = (rmag >=19.5) & (rmag <= 21)


    bgs_mask = (  MASKBITS & QC & MAGLIM )    

    return sweep[bgs_mask].to_pandas()


@dask.delayed
def read_catalogs(paths, survey):
    dfs = list()
    for path in paths:
        with fitsio.FITS(path) as fits:
            hdu = fits[1]
            nrows = hdu.get_nrows()
            for start in range(0, nrows, ROW_CHUNK):
                sweep = Table(hdu[start:start+ROW_CHUNK], copy=False)
                dfs.append(select_targets(sweep, survey))
    return pd.concat(dfs, ignore_index=True)


In [6]:
# Chunk size is the number of files each worker opens
chunksize = 4
parts = list()

survey = "south" # choose from `north` or `south
sweep_path = cat_path / survey / "sweep" / "11.0"
sweep_files = list(sweep_path.glob("sweep*"))
for i in range(0, len(sweep_files), chunksize):
    parts.append(read_catalogs(sweep_files[i:i+chunksize], survey))

# survey = "north" # choose from `north` or `south
# sweep_path = cat_path / survey / "sweep" / "11.0"
# sweep_files = list(sweep_path.glob("sweep*"))
# for i in range(0, len(sweep_files), chunksize):
#     parts.append(read_catalogs(sweep_files[i:i+chunksize], survey))

In [7]:
dd = dask.dataframe.from_delayed(parts)
dd = dd.set_index("MYID")
dd = dd.repartition(npartitions=2048)
dd = dd.persist()

In [8]:
filepath = dest_path / f"bgs_ext_target_list_dr11_ALL.parquet"
dd.to_parquet(filepath, compression='gzip', engine='pyarrow',overwrite=True,
              write_metadata_file=True)

In [9]:
print("Total number of Objects:", len(dd)/1e6)

Total number of Objects: 257.218464


### READ and save main survey TARGETIDs

In [1]:
from astropy.table import Table, hstack, vstack
import fitsio
from pathlib import Path
from tqdm import tqdm
from joblib import Parallel, delayed
import os
import pandas as pd

In [2]:
main_survey_targets = Path("/global/cfs/cdirs/desi/survey/ops/surveyops/trunk/mtl/main/")
main_survey_programs = ["dark", "bright", "dark1b", "bright1b"]
dest_path = Path(os.environ["SCRATCH"]) / "photo_z_spare_fiber"

In [3]:
def read_target_list(fp):
    targets = Table.read(fp, format="ascii.ecsv")
    return targets[["RA","DEC", "DESI_TARGET", "TARGETID"]]

In [4]:
all_targets = []
for program in main_survey_programs:
    print(f"processing {program} program")

    table_list = Parallel(n_jobs=100)(
    delayed(read_target_list)(fp) 
    for fp in tqdm((main_survey_targets / program ).glob("*.ecsv")))
    all_targets.append(vstack(table_list, metadata_conflicts="silent").to_pandas())

processing dark program


9303it [00:35, 258.82it/s]


processing bright program


9518it [00:39, 238.60it/s]


processing dark1b program


9171it [00:04, 1903.99it/s]


processing bright1b program


1797it [00:01, 1491.47it/s]


In [5]:
all_targets = pd.concat(all_targets)
all_targets.to_parquet( dest_path / "all_main_survey_targets.parquet")
print("Total number of targets: ", len(all_targets)/1e6)

Total number of targets:  405.969501


### Resample the data and remove main survey targets

In [1]:
from pathlib import Path
import os
import pandas as pd
import fitsio
from astropy.table import Table
import numpy as np
from desitarget.targetmask import desi_mask
from desimodel.footprint import is_point_in_desi
import pandas as pd

In [2]:
dest_path = Path(os.environ["SCRATCH"]) / "photo_z_spare_fiber"

my_targets = pd.read_parquet(dest_path / f"bgs_ext_target_list_dr11_ALL.parquet",
                     columns = ["MYID",'RA','DEC','TYPE', 'RELEASE', 'BRICKID', 'OBJID', 'PMRA', 'PMDEC', 'REF_EPOCH', 'FLUX_G', 'FLUX_R', 'FLUX_Z'])
my_targets = my_targets.reset_index()

desi_targets = pd.read_parquet(dest_path / "all_main_survey_targets.parquet")

In [3]:
desi_tgt = desi_targets["DESI_TARGET"]

is_bgs  = (desi_tgt & desi_mask.BGS_ANY != 0)   #- instead of 2**60
overlaps = np.isin(my_targets["MYID"], desi_targets["TARGETID"][is_bgs])
print(f"Overlap Fraction with BGS_ANY:{overlaps.sum()*100/len(overlaps)}")

is_lrg  = (desi_tgt & desi_mask.LRG != 0)
overlaps = np.isin(my_targets["MYID"], desi_targets["TARGETID"][is_lrg])
print(f"Overlap Fraction with LRG:{overlaps.sum()*100/len(overlaps)}")

is_elg  = (desi_tgt & desi_mask.ELG != 0)
overlaps = np.isin(my_targets["MYID"], desi_targets["TARGETID"][is_elg])
print(f"Overlap Fraction with ELG:{overlaps.sum()*100/len(overlaps)}")

is_qso  = (desi_tgt & desi_mask.QSO != 0)
overlaps = np.isin(my_targets["MYID"], desi_targets["TARGETID"][is_qso])
print(f"Overlap Fraction with QSO:{overlaps.sum()*100/len(overlaps)}")

is_mws  = (desi_tgt & desi_mask.MWS_ANY != 0)
overlaps = np.isin(my_targets["MYID"], desi_targets["TARGETID"][is_mws])
print(f"Overlap Fraction with MWS_ANY:{overlaps.sum()*100/len(overlaps)}")

is_lge = (desi_tgt & desi_mask.LGE != 0)
overlaps = np.isin(my_targets["MYID"], desi_targets["TARGETID"][is_lge])
print(f"Overlap Fraction with LGE:{overlaps.sum()*100/len(overlaps)}")


is_psf = (my_targets['TYPE'] == "PSF")
print(f"Fraction of PSF objects:{is_psf.sum()*100/len(is_psf)}")

Overlap Fraction with BGS_ANY:3.061637130373347
Overlap Fraction with LRG:0.7603793170928818
Overlap Fraction with ELG:0.28038578132555836
Overlap Fraction with QSO:0.5326425555515331
Overlap Fraction with MWS_ANY:2.816054060566974
Overlap Fraction with LGE:0.40836531859548
Fraction of PSF objects:62.47706579882228


In [4]:
# remove PSF type objects
my_targets = my_targets[my_targets["TYPE"]!="PSF"]
len(my_targets)

96515915

In [5]:
my_targets = my_targets.reset_index()

# Ensure things are only in DR11 footprint

In [6]:
ts = Table.read("/global/cfs/cdirs/desi/survey/ops/surveyops/trunk/ops/tiles-specstatus.ecsv")
stids = set(ts[(ts["LASTNIGHT"] < 20260514) & (ts["FAPRGRM"] == "dark") & (ts["SURVEY"] == "main")]["TILEID"])
t = Table.read("/global/cfs/cdirs/desi/survey/ops/surveyops/trunk/ops/tiles-main.ecsv")
ii = (t["PROGRAM"] == "DARK") & t["IN_DESI"]
tdark = t[ii]
ii = np.array([tid not in stids for tid in tdark["TILEID"]])
remaining_dark_tiles = tdark[ii]

In [7]:
ii = is_point_in_desi(remaining_dark_tiles, my_targets["RA"], my_targets["DEC"])

In [8]:
my_targets = my_targets[ii]
my_targets = my_targets.reset_index()

# Resample

In [9]:
rng = np.random.default_rng(299792458)

resampled_my_targets = my_targets.sample(frac = 0.05, random_state=rng)

In [10]:
resampled_my_targets.to_parquet(dest_path / "bgs_ext_target_list_dr11_DOWNSAMPLED.parquet")

In [11]:
is_bgs  = (desi_tgt & desi_mask.BGS_ANY != 0)   #- instead of 2**60
overlaps = np.isin(resampled_my_targets["MYID"], desi_targets["TARGETID"][is_bgs])
print(f"Overlap Number with BGS_ANY:{overlaps.sum()}")
print(f"Overlap Fraction with BGS_ANY:{overlaps.sum()*100/len(overlaps)}")

is_lrg  = (desi_tgt & desi_mask.LRG != 0)
overlaps = np.isin(resampled_my_targets["MYID"], desi_targets["TARGETID"][is_lrg])
print(f"Overlap Number with LRG:{overlaps.sum()}")
print(f"Overlap Fraction with LRG:{overlaps.sum()*100/len(overlaps)}")

is_elg  = (desi_tgt & desi_mask.ELG != 0)
overlaps = np.isin(resampled_my_targets["MYID"], desi_targets["TARGETID"][is_elg])
print(f"Overlap Number with ELG:{overlaps.sum()}")
print(f"Overlap Fraction with ELG:{overlaps.sum()*100/len(overlaps)}")

is_qso  = (desi_tgt & desi_mask.QSO != 0)
overlaps = np.isin(resampled_my_targets["MYID"], desi_targets["TARGETID"][is_qso])
print(f"Overlap Number with QSO:{overlaps.sum()}")
print(f"Overlap Fraction with QSO:{overlaps.sum()*100/len(overlaps)}")

is_mws  = (desi_tgt & desi_mask.MWS_ANY != 0)
overlaps = np.isin(resampled_my_targets["MYID"], desi_targets["TARGETID"][is_mws])
print(f"Overlap Number with MWS_ANY:{overlaps.sum()}")
print(f"Overlap Fraction with MWS_ANY:{overlaps.sum()*100/len(overlaps)}")

is_lge = (desi_tgt & desi_mask.LGE != 0)
overlaps = np.isin(resampled_my_targets["MYID"], desi_targets["TARGETID"][is_lge])
print(f"Overlap Number with LGE:{overlaps.sum()}")
print(f"Overlap Fraction with LGE:{overlaps.sum()*100/len(overlaps)}")



Overlap Number with BGS_ANY:68330
Overlap Fraction with BGS_ANY:10.659690209090924
Overlap Number with LRG:22090
Overlap Fraction with LRG:3.446107957248917
Overlap Number with ELG:1597
Overlap Fraction with ELG:0.24913691297992396
Overlap Number with QSO:0
Overlap Fraction with QSO:0.0
Overlap Number with MWS_ANY:0
Overlap Fraction with MWS_ANY:0.0
Overlap Number with LGE:13555
Overlap Fraction with LGE:2.1146217003399306


In [12]:
# remove BGS_ANY
mask = ~np.isin(resampled_my_targets["MYID"],desi_targets["TARGETID"][is_bgs])
final_my_targets = resampled_my_targets[mask]
print(f"{(~mask).sum()} BGS_ANY removed")

# remove LRG
mask = ~np.isin(final_my_targets["MYID"],desi_targets["TARGETID"][is_lrg])
final_my_targets = final_my_targets[mask]
print(f"{(~mask).sum()} LRG removed")

# remove ELG
mask = ~np.isin(final_my_targets["MYID"],desi_targets["TARGETID"][is_elg])
final_my_targets = final_my_targets[mask]
print(f"{(~mask).sum()} ELG removed")

# remove QSO
mask = ~np.isin(final_my_targets["MYID"],desi_targets["TARGETID"][is_qso])
final_my_targets = final_my_targets[mask]
print(f"{(~mask).sum()} QSO removed")

# remove LGE
mask = ~np.isin(final_my_targets["MYID"],desi_targets["TARGETID"][is_lge])
final_my_targets = final_my_targets[mask]
print(f"{(~mask).sum()} LGE removed")

68330 BGS_ANY removed
18134 LRG removed
1389 ELG removed
0 QSO removed
11503 LGE removed


In [13]:
print(f"Final target List length:{len(final_my_targets)/1e6}")

Final target List length:0.541657


In [14]:
final_my_targets.to_parquet(dest_path / "bgs_ext_target_list_dr11.parquet")

# Comply with the pipeline format

In [15]:
final_my_targets = Table.from_pandas(final_my_targets)
final_my_targets["OVERRIDE"] = False

In [16]:
final_my_targets.dtype

dtype([('level_0', '<i8'), ('index', '<i8'), ('MYID', '<i8'), ('RA', '<f8'), ('DEC', '<f8'), ('TYPE', '<U3'), ('RELEASE', '<i2'), ('BRICKID', '<i4'), ('OBJID', '<i4'), ('PMRA', '<f4'), ('PMDEC', '<f4'), ('REF_EPOCH', '<f4'), ('FLUX_G', '<f4'), ('FLUX_R', '<f4'), ('FLUX_Z', '<f4'), ('OVERRIDE', '?')])

In [17]:
arr = final_my_targets.as_array()
flipped_byte_order_cat = Table(arr.view(arr.dtype.newbyteorder()).byteswap())
print(flipped_byte_order_cat.dtype)

[('level_0', '>i8'), ('index', '>i8'), ('MYID', '>i8'), ('RA', '>f8'), ('DEC', '>f8'), ('TYPE', '>U3'), ('RELEASE', '>i2'), ('BRICKID', '>i4'), ('OBJID', '>i4'), ('PMRA', '>f4'), ('PMDEC', '>f4'), ('REF_EPOCH', '>f4'), ('FLUX_G', '>f4'), ('FLUX_R', '>f4'), ('FLUX_Z', '>f4'), ('OVERRIDE', '?')]


In [18]:
flipped_byte_order_cat

level_0,index,MYID,RA,DEC,TYPE,RELEASE,BRICKID,OBJID,PMRA,PMDEC,REF_EPOCH,FLUX_G,FLUX_R,FLUX_Z,OVERRIDE
int64,int64,int64,float64,float64,str3,int16,int32,int32,float32,float32,float32,float32,float32,float32,bool
13720285,36506067,48423454847801664,320.3680027178066,-17.961063936659478,SER,11010,229540,6464,0.0,0.0,0.0,1.4595191,4.04434,7.129249,False
20269169,54224719,48423577346637961,322.107288179504,-12.854298372220157,SER,11010,258746,137,0.0,0.0,0.0,2.9843879,7.2894235,8.830059,False
19559485,52228669,48423563442520509,192.57784567164316,-13.207088556421196,REX,11010,255431,445,0.0,0.0,2016.0,1.4927146,7.4894195,17.243244,False
7627818,20291216,48423336815886889,18.71611897084039,-22.896887867426425,SER,11010,201399,553,0.0,0.0,0.0,2.7293687,5.710946,9.382835,False
936352,2477646,48423201583145537,44.6147150422971,-29.1306334814982,SER,11010,169157,8769,0.0,0.0,0.0,2.5587788,4.424265,6.3983326,False
19535607,52189474,48423562951789485,162.69337991900795,-13.168084289313265,DEV,11010,255314,2989,0.0,0.0,0.0,0.87005997,4.8299246,6.442209,False
16577285,44387350,48423505477242175,228.63028378572906,-15.842491058228168,EXP,11010,241611,3391,0.0,0.0,0.0,2.5335095,4.120637,5.4932547,False
4940377,13379538,48423276094954099,8.467641318046336,-25.72645262191968,REX,11010,186922,6771,0.0,0.0,0.0,4.590516,8.67185,14.754353,False
6670768,17812058,48423315072617171,46.784209288274106,-23.877915872136704,REX,11010,196215,2771,0.0,0.0,0.0,8.833942,14.562854,20.268536,False


In [19]:
flipped_byte_order_cat.write("/global/cfs/cdirs/desi/target/proposals/spare_proposals_Y5/indata/BGS_EXT.fits", overwrite=True)

In [20]:
from desitarget.secondary import check_file_format
check_file_format("/global/cfs/cdirs/desi/target/proposals/spare_proposals_Y5/indata/BGS_EXT.fits", new_program=False)

True